In [1]:
%display latex

**Name.** Juan Pablo Otálvaro Ghisays

# Einstein-Rosen Waves — Pseudotensor de Landau-Lifshitz

Métrica en coordenadas cilíndricas $(t, \rho, \phi, z)$:
$$ds^2 = e^{2(\gamma-\psi)}(dt^2 - d\rho^2) - \rho^2 e^{-2\psi}\,d\phi^2 - e^{2\psi}\,dz^2$$

El pseudotensor de Landau-Lifshitz se define como:
$$t^{\mu\nu}_{\scriptscriptstyle\rm LL} = \frac{c^4}{16\pi G}\frac{1}{(-g)}
\partial_\rho\partial_\sigma\!\left[(-g)\left(g^{\mu\nu}g^{\rho\sigma}-g^{\mu\rho}g^{\nu\sigma}\right)\right] - T^{\mu\nu}$$

En vacío $T^{\mu\nu}=0$, de modo que:
$$t^{\mu\nu}_{\scriptscriptstyle\rm LL} = \frac{c^4}{16\pi G}\frac{1}{(-g)}
\partial_\rho\partial_\sigma\!\left[(-g)\left(g^{\mu\nu}g^{\rho\sigma}-g^{\mu\rho}g^{\nu\sigma}\right)\right]$$

## 1. Variedad y carta de coordenadas

In [2]:
M = Manifold(4, 'M', r'\mathcal{M}')
U = M.open_subset('U')

C.<t, rh, ph, z> = U.chart(r't:(-oo,+oo) rh:(0,+oo):\rho ph:(0,2*pi):\phi z:(-oo,+oo)')
C

Chart (U, (t, rh, ph, z))

## 2. Funciones y métrica

In [5]:
# Funciones arbitrarias psi(t, rho), chi(t, rho), k(t, rho)
# Las pasamos como funciones de (t, x, y) via rho = sqrt(x^2+y^2)
psi_f = function('psi', nargs=2)
chi_f = function('chi', nargs=2)
#k_f   = function('k',   nargs=2)

PSI = psi_f(t, rh)
CHI = chi_f(t, rh)

g = M.lorentzian_metric('g')

g[0,0], g[1,1], g[2,2], g[3,3] = exp(2*(CHI-PSI) ), -exp(2*(CHI-PSI ) ),-rh^2*exp(-2*PSI),-exp(2*PSI)

g.display()

g = e^(2*chi(t, rh) - 2*psi(t, rh)) dt⊗dt - e^(2*chi(t, rh) - 2*psi(t, rh)) drh⊗drh - rh^2*e^(-2*psi(t, rh)) dph⊗dph - e^(2*psi(t, rh)) dz⊗dz

## 3. Métrica inversa y determinante

In [6]:
n    = 4
crds = [t, rh, ph, z]
cn   = ['t', 'rh', 'ph', 'z']

gI  = g.inverse()
gUU = [[gI[i,j].expr() for j in range(n)] for i in range(n)]
gLL = matrix([[g[i,j].expr() for j in range(n)] for i in range(n)])

det_g  = gLL.det().simplify_full().factor()
neg_g  = (-det_g).simplify_full().factor()   # (-g) > 0

print('det(g) =')
show(det_g)
print('(-g) =')
show(neg_g)

det(g) =


-rh^2*e^(4*chi(t, rh) - 4*psi(t, rh))

(-g) =


rh^2*e^(4*chi(t, rh) - 4*psi(t, rh))

## 4. Tensor de Landau-Lifshitz $\mathfrak{H}^{\mu\nu\rho\sigma}$

Definimos la cantidad entre corchetes:
$$\mathfrak{H}^{\mu\nu\rho\sigma} \equiv (-g)\left(g^{\mu\nu}g^{\rho\sigma} - g^{\mu\rho}g^{\nu\sigma}\right)$$

El pseudotensor queda entonces:
$$t^{\mu\nu}_{\scriptscriptstyle\rm LL} = \frac{c^4}{16\pi G}\frac{1}{(-g)}\partial_\rho\partial_\sigma\,\mathfrak{H}^{\mu\nu\rho\sigma}$$

In [7]:
# H[mu][nu][rho][sigma] = (-g)(g^{munu}g^{rhosigma} - g^{murho}g^{nusigma})
print('Construyendo H^{munurhosigma}...')

H = [[[[SR(0)]*n for _ in range(n)] for _ in range(n)] for _ in range(n)]

for mu in range(n):
    for nu in range(n):
        for rho in range(n):
            for sigma in range(n):
                val = neg_g * (gUU[mu][nu]*gUU[rho][sigma] - gUU[mu][rho]*gUU[nu][sigma])
                H[mu][nu][rho][sigma] = val.simplify_full()

print('H^{munurhosigma} construido.')

# Mostrar algunas componentes no nulas como verificacion
print('\nEjemplos de componentes no nulas H^{munurhosigma}:')
shown = 0
for mu in range(n):
    for nu in range(mu, n):
        for rho in range(n):
            for sigma in range(rho, n):
                val = H[mu][nu][rho][sigma]
                if val != 0 and shown < 4:
                    print(f'  H^{{{cn[mu]}{cn[nu]}{cn[rho]}{cn[sigma]}}} =')
                    show(val.factor())
                    shown += 1

Construyendo H^{munurhosigma}...
H^{munurhosigma} construido.

Ejemplos de componentes no nulas H^{munurhosigma}:
  H^{ttrhrh} =


-rh^2

  H^{ttphph} =


-e^(2*chi(t, rh))

  H^{ttzz} =


-rh^2*e^(2*chi(t, rh) - 4*psi(t, rh))

  H^{trhtrh} =


rh^2

## 5. Segunda derivada $\partial_\rho\partial_\sigma \mathfrak{H}^{\mu\nu\rho\sigma}$

In [8]:
# ddH[mu][nu] = sum_{rho,sigma} d^2 H^{munu rho sigma} / (dx^rho dx^sigma)
print('Calculando sum_{rho,sigma} partial_rho partial_sigma H^{munurhosigma}...')
print('(puede tardar unos minutos)')

ddH = [[SR(0)]*n for _ in range(n)]

for mu in range(n):
    for nu in range(mu, n):   # simetria munu
        acum = SR(0)
        for rho in range(n):
            for sigma in range(n):
                acum += diff(diff(H[mu][nu][rho][sigma], crds[rho]), crds[sigma])
        val = acum.simplify_full()
        ddH[mu][nu] = val
        ddH[nu][mu] = val   # simetria
    print(f'  fila mu={cn[mu]} lista.')

print('Segunda derivada calculada.')

Calculando sum_{rho,sigma} partial_rho partial_sigma H^{munurhosigma}...
(puede tardar unos minutos)
  fila mu=t lista.
  fila mu=rh lista.
  fila mu=ph lista.
  fila mu=z lista.
Segunda derivada calculada.


## 6. Pseudotensor de Landau-Lifshitz $t^{\mu\nu}_{\rm LL}$

$$t^{\mu\nu}_{\scriptscriptstyle\rm LL} = \frac{c^4}{16\pi G}\frac{1}{(-g)}\,\sum_{\rho,\sigma}\partial_\rho\partial_\sigma\mathfrak{H}^{\mu\nu\rho\sigma}$$

In [9]:
c_light, G_N = var('c G_N', domain='positive')
prefactor = c_light^4 / (16 * pi * G_N)

t_LL = [[SR(0)]*n for _ in range(n)]

for mu in range(n):
    for nu in range(mu, n):
        val = (prefactor * ddH[mu][nu] / neg_g).simplify_full()
        t_LL[mu][nu] = val
        t_LL[nu][mu] = val

print('Pseudotensor de Landau-Lifshitz calculado.')

Pseudotensor de Landau-Lifshitz calculado.


## 7. Resultados

In [10]:
for mu in range(n):
    for nu in range(mu, n):
        val = t_LL[mu][nu]
        lbl = f't^{{\\scriptscriptstyle\\rm LL}},{{{cn[mu]}{cn[nu]}}}'
        if val == 0:
            print(f't_LL^{{{cn[mu]}{cn[nu]}}} = 0')
        else:
            print(f't_LL^{{{cn[mu]}{cn[nu]}}} =')
            show(val.factor())

t_LL^{tt} =


-1/8*c^4*e^(-4*chi(t, rh) + 4*psi(t, rh))/(pi*G_N*rh^2)

t_LL^{trh} = 0
t_LL^{tph} = 0
t_LL^{tz} = 0
t_LL^{rhrh} = 0
t_LL^{rhph} = 0
t_LL^{rhz} = 0
t_LL^{phph} =


-1/8*(2*diff(chi(t, rh), t)^2 - 2*diff(chi(t, rh), rh)^2 + diff(chi(t, rh), t, t) - diff(chi(t, rh), rh, rh))*c^4*e^(-2*chi(t, rh) + 4*psi(t, rh))/(pi*G_N*rh^2)

t_LL^{phz} = 0
t_LL^{zz} =


-1/8*(2*rh^2*diff(chi(t, rh), t)^2 - 2*rh^2*diff(chi(t, rh), rh)^2 - 8*rh^2*diff(chi(t, rh), t)*diff(psi(t, rh), t) + 8*rh^2*diff(psi(t, rh), t)^2 + 8*rh^2*diff(chi(t, rh), rh)*diff(psi(t, rh), rh) - 8*rh^2*diff(psi(t, rh), rh)^2 + rh^2*diff(chi(t, rh), t, t) - rh^2*diff(chi(t, rh), rh, rh) - 2*rh^2*diff(psi(t, rh), t, t) + 2*rh^2*diff(psi(t, rh), rh, rh) - 4*rh*diff(chi(t, rh), rh) + 8*rh*diff(psi(t, rh), rh) - 1)*c^4*e^(-2*chi(t, rh))/(pi*G_N*rh^2)

## 8. Verificaciones

In [11]:
# 1. Simetria t^{munu} = t^{numu}  (propiedad definitoria de LL vs Tolman)
print('Verificacion simetria t_LL^{munu} = t_LL^{numu}:')
sim_ok = all((t_LL[mu][nu] - t_LL[nu][mu]).simplify_full() == 0
             for mu in range(n) for nu in range(n))
print('  OK' if sim_ok else '  FALLO')

# 2. Simetria de H en cada par de indices por separado
#    H^{munu rs} = (-g)(g^{munu}g^{rs} - g^{mur}g^{nus})
#    es simetrico en mu<->nu y en rho<->sigma por separado,
#    y ademas simetrico bajo intercambio del par (mu,nu)<->(rho,sigma).
print('\nVerificacion simetria H^{munu rs} = H^{numu rs}:')
sym_mn = all((H[mu][nu][rho][sigma] - H[nu][mu][rho][sigma]).simplify_full() == 0
             for mu in range(n) for nu in range(n)
             for rho in range(n) for sigma in range(n))
print('  OK' if sym_mn else '  FALLO')

print('\nVerificacion simetria H^{mn rho sigma} = H^{mn sigma rho}:')
sym_rs = all((H[mu][nu][rho][sigma] - H[mu][nu][sigma][rho]).simplify_full() == 0
             for mu in range(n) for nu in range(n)
             for rho in range(n) for sigma in range(n))
print('  OK' if sym_rs else '  FALLO')

print('\nVerificacion simetria bajo intercambio de pares H^{munu rs} = H^{rs munu}:')
sym_pair = all((H[mu][nu][rho][sigma] - H[rho][sigma][mu][nu]).simplify_full() == 0
               for mu in range(n) for nu in range(n)
               for rho in range(n) for sigma in range(n))
print('  OK' if sym_pair else '  FALLO')

# 3. Traza g_{munu} t_LL^{munu}
print('\nTraza g_{munu} t_LL^{munu} =')
gLL_expr = [[g[i,j].expr() for j in range(n)] for i in range(n)]
traza = sum(gLL_expr[mu][nu] * t_LL[mu][nu]
            for mu in range(n) for nu in range(n)).simplify_full()
show(traza)

Verificacion simetria t_LL^{munu} = t_LL^{numu}:
  OK

Verificacion simetria H^{munu rs} = H^{numu rs}:
  FALLO

Verificacion simetria H^{mn rho sigma} = H^{mn sigma rho}:
  FALLO

Verificacion simetria bajo intercambio de pares H^{munu rs} = H^{rs munu}:
  OK

Traza g_{munu} t_LL^{munu} =


1/4*(2*c^4*rh^2*e^(2*psi(t, rh))*diff(chi(t, rh), t)^2 - 2*c^4*rh^2*e^(2*psi(t, rh))*diff(chi(t, rh), rh)^2 - 4*c^4*rh^2*e^(2*psi(t, rh))*diff(chi(t, rh), t)*diff(psi(t, rh), t) + 4*c^4*rh^2*e^(2*psi(t, rh))*diff(psi(t, rh), t)^2 - 4*c^4*rh^2*e^(2*psi(t, rh))*diff(psi(t, rh), rh)^2 + c^4*rh^2*e^(2*psi(t, rh))*diff(chi(t, rh), t, t) - c^4*rh^2*e^(2*psi(t, rh))*diff(chi(t, rh), rh, rh) - c^4*rh^2*e^(2*psi(t, rh))*diff(psi(t, rh), t, t) + c^4*rh^2*e^(2*psi(t, rh))*diff(psi(t, rh), rh, rh) - 2*c^4*rh*e^(2*psi(t, rh))*diff(chi(t, rh), rh) - c^4*e^(2*psi(t, rh)) + 4*(c^4*rh^2*e^(2*psi(t, rh))*diff(chi(t, rh), rh) + c^4*rh*e^(2*psi(t, rh)))*diff(psi(t, rh), rh))*e^(-2*chi(t, rh))/(pi*G_N*rh^2)

In [ ]:
# Tensor de Einstein como referencia (vacio => G_munu = 0)
ric = g.ricci()
r   = g.ricci_scalar()
G   = ric - (1/2)*g*r
print('Tensor de Einstein G_munu:')
G.display()

## 9. Verificación de la ley de conservación

En vacío $T^{\mu\nu}=0$, la ley de conservación del pseudotensor de Landau-Lifshitz es:
$$\partial_\nu\left[(-g)\,t^{\mu\nu}_{\rm LL}\right] = 0 \qquad \forall\,\mu$$

que es equivalente a verificar:
$$\partial_\nu\!\left[\partial_\rho\partial_\sigma\,\mathfrak{H}^{\mu\nu\rho\sigma}\right] = 0$$

ya que $(-g)$ no cancela con la derivada exterior. Comprobamos ambas formas.

In [12]:
# Ley de conservacion: partial_nu [ (-g) t_LL^{munu} ] = 0
# Equivalentemente: partial_nu [ ddH[mu][nu] / (-g) * (-g) ]
#                 = partial_nu [ ddH[mu][nu] ] = 0
# (el prefactor c^4/16piG es constante)

print('Verificacion ley de conservacion partial_nu[(-g) t_LL^{munu}] = 0')
print('='*60)

conserved = True
for mu in range(n):
    div = SR(0)
    for nu in range(n):
        # (-g) * t_LL^{munu}  =  prefactor * ddH[mu][nu]
        # derivamos prefactor * ddH[mu][nu] respecto a x^nu
        div += diff((neg_g * t_LL[mu][nu] / prefactor).simplify_full(), crds[nu])
    res = div.simplify_full()
    status = 'OK (= 0)' if res == 0 else f'FALLO: {res}'
    print(f'  mu = {cn[mu]}: partial_nu[(-g) t_LL^{{{cn[mu]}nu}}] = ', end='')
    if res == 0:
        print('0  ->  OK')
    else:
        conserved = False
        print('FALLO')
        show(res.factor())

print()
if conserved:
    print('Ley de conservacion verificada para todos los indices mu.')
else:
    print('Alguna componente no se conserva: revisar calculo.')

Verificacion ley de conservacion partial_nu[(-g) t_LL^{munu}] = 0
  mu = t: partial_nu[(-g) t_LL^{tnu}] = 0  ->  OK
  mu = rh: partial_nu[(-g) t_LL^{rhnu}] = 0  ->  OK
  mu = ph: partial_nu[(-g) t_LL^{phnu}] = 0  ->  OK
  mu = z: partial_nu[(-g) t_LL^{znu}] = 0  ->  OK

Ley de conservacion verificada para todos los indices mu.


In [ ]:
# Exportar LaTeX
with open('resultados_LL.tex', 'w') as f:
    f.write('% Pseudotensor de Landau-Lifshitz -- Einstein-Rosen cilindricas\n% SageMath\n\n')
    f.write('\\section*{Pseudotensor de Landau-Lifshitz $t^{\\mu\\nu}_{\\rm LL}$}\n')
    f.write('\\begin{align}\n')
    for mu in range(n):
        for nu in range(mu, n):
            val = t_LL[mu][nu]
            entry = latex(val.factor()) if val != 0 else '0'
            f.write(f't^{{{cn[mu]}{cn[nu]}}} &= {entry} \\\\\n')
    f.write('\\end{align}\n')
print('Archivo resultados_LL.tex generado.')